# 🌳 Day 04a: Decision Trees & Random Forest — From Entropy to Ensemble

---

## 🎯 What You'll Master Today
- Entropy, information gain, Gini impurity
- Decision tree from scratch (ID3 algorithm)
- Random Forest: bagging + feature sampling
- sklearn implementation

**Goal:** Calculate information gain by hand AND code a decision tree from scratch.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  DECISION TREE — SPLITTING CRITERIA                              ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Entropy:    H(S) = -Σ pᵢ log₂(pᵢ)                            ║
║              H=0 → pure, H=1 → max uncertainty (binary)        ║
║                                                                   ║
║  Info Gain:  IG(S,A) = H(S) - Σ (|Sᵥ|/|S|) · H(Sᵥ)          ║
║              Higher IG → better split                            ║
║                                                                   ║
║  Gini:       G(S) = 1 - Σ pᵢ²                                  ║
║              G=0 → pure, G=0.5 → max impurity (binary)        ║
║                                                                   ║
║  RANDOM FOREST = Bagging + Feature Sampling                     ║
║    • Train N trees on bootstrap samples                         ║
║    • Each split considers √features (not all)                   ║
║    • Majority vote for prediction                                ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np
from collections import Counter

# ============================================================
# Entropy & Gini — The Math
# ============================================================

def entropy(y):
    """H(S) = -Σ pᵢ log₂(pᵢ)"""
    counts = np.bincount(y)
    probs = counts[counts > 0] / len(y)
    return -np.sum(probs * np.log2(probs))

def gini(y):
    """G(S) = 1 - Σ pᵢ²"""
    counts = np.bincount(y)
    probs = counts[counts > 0] / len(y)
    return 1 - np.sum(probs ** 2)

# Examples
pure = np.array([1, 1, 1, 1])          # all same class
mixed = np.array([1, 1, 0, 0])          # 50-50
mostly = np.array([1, 1, 1, 0])         # 75-25

print(f"{'Dataset':<15} {'Entropy':>8} {'Gini':>8}")
print(f"{'Pure [1,1,1,1]':<15} {entropy(pure):>8.3f} {gini(pure):>8.3f}")
print(f"{'Mixed [1,1,0,0]':<15} {entropy(mixed):>8.3f} {gini(mixed):>8.3f}")
print(f"{'Mostly [1,1,1,0]':<15} {entropy(mostly):>8.3f} {gini(mostly):>8.3f}")

In [ ]:
# ============================================================
# Information Gain Calculation (by hand then code)
# ============================================================

# Example: Should I play tennis?
# Labels: [Y,Y,Y,N,N,Y,N,Y,Y,N]  → 6Y, 4N
y_parent = np.array([1,1,1,0,0,1,0,1,1,0])

# Split by "Outlook=Sunny": Left=[Y,N,N], Right=[Y,Y,Y,Y,N,N,Y]
y_left = np.array([1,0,0])     # Sunny
y_right = np.array([1,1,1,1,0,0,1])  # Not Sunny

def information_gain(y_parent, y_left, y_right):
    n = len(y_parent)
    n_left, n_right = len(y_left), len(y_right)
    
    parent_entropy = entropy(y_parent)
    weighted_child = (n_left/n) * entropy(y_left) + (n_right/n) * entropy(y_right)
    
    return parent_entropy - weighted_child

ig = information_gain(y_parent, y_left, y_right)
print(f"Parent entropy: {entropy(y_parent):.4f}")
print(f"Left entropy:   {entropy(y_left):.4f} (weight={len(y_left)/len(y_parent):.2f})")
print(f"Right entropy:  {entropy(y_right):.4f} (weight={len(y_right)/len(y_parent):.2f})")
print(f"Information Gain: {ig:.4f}")
print(f"\n💡 Higher IG → better split. Choose the feature with highest IG.")

In [ ]:
# ============================================================
# ⭐ Decision Tree — From Scratch
# ============================================================

class DecisionTreeScratch:
    def __init__(self, max_depth=10, min_samples=2):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.tree = None
    
    def _best_split(self, X, y):
        best_ig = -1
        best_feature, best_threshold = None, None
        
        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                
                ig = information_gain(y, y[left_mask], y[right_mask])
                if ig > best_ig:
                    best_ig = ig
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_ig
    
    def _build_tree(self, X, y, depth=0):
        # Stopping conditions
        if (depth >= self.max_depth or 
            len(y) < self.min_samples or 
            len(np.unique(y)) == 1):
            return Counter(y).most_common(1)[0][0]
        
        feature, threshold, ig = self._best_split(X, y)
        if feature is None or ig == 0:
            return Counter(y).most_common(1)[0][0]
        
        left_mask = X[:, feature] <= threshold
        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[~left_mask], y[~left_mask], depth + 1)
        
        return {'feature': feature, 'threshold': threshold,
                'left': left, 'right': right}
    
    def fit(self, X, y):
        self.tree = self._build_tree(X, y)
        return self
    
    def _predict_one(self, x, node):
        if not isinstance(node, dict):
            return node
        if x[node['feature']] <= node['threshold']:
            return self._predict_one(x, node['left'])
        return self._predict_one(x, node['right'])
    
    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])

# Test
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=200, n_features=4, random_state=42)
X_train, X_test = X[:160], X[160:]
y_train, y_test = y[:160], y[160:]

tree = DecisionTreeScratch(max_depth=5)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)
print(f"Scratch tree accuracy: {np.mean(y_pred == y_test):.3f}")

In [ ]:
# ============================================================
# sklearn: Decision Tree + Random Forest
# ============================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
print(f"Decision Tree acc: {dt.score(X_test, y_test):.3f}")

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
print(f"Random Forest acc: {rf.score(X_test, y_test):.3f}")

# Feature importance
print(f"\nFeature importance: {rf.feature_importances_.round(3)}")
for i, imp in enumerate(rf.feature_importances_):
    print(f"  Feature {i}: {imp:.3f} {'█' * int(imp * 30)}")

print("\n💡 RF almost always beats single tree. Default go-to for tabular data.")

In [ ]:
# ============================================================
# Pruning & Hyperparameters
# ============================================================

from sklearn.model_selection import cross_val_score

# Key hyperparameters
configs = [
    {'max_depth': 3},
    {'max_depth': 5},
    {'max_depth': 10},
    {'max_depth': None},  # no limit
]

print("Depth effect on Decision Tree:")
for cfg in configs:
    dt = DecisionTreeClassifier(**cfg, random_state=42)
    scores = cross_val_score(dt, X, y, cv=5)
    print(f"  depth={str(cfg['max_depth']):>4s}: CV acc={scores.mean():.3f} ± {scores.std():.3f}")

print("\nRandom Forest hyperparameters:")
rf_configs = [
    {'n_estimators': 10},
    {'n_estimators': 50},
    {'n_estimators': 100},
    {'n_estimators': 200},
]
for cfg in rf_configs:
    rf = RandomForestClassifier(**cfg, max_depth=5, random_state=42)
    scores = cross_val_score(rf, X, y, cv=5)
    print(f"  trees={cfg['n_estimators']:>3d}: CV acc={scores.mean():.3f} ± {scores.std():.3f}")

---

## 🗺️ Trees & Random Forest Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  TREES — Decision Guide                                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  SPLITTING:                                                      ║
║    Entropy → IG = H(parent) - weighted_H(children)              ║
║    Gini → sklearn default for classification                    ║
║                                                                   ║
║  OVERFITTING CONTROL:                                            ║
║    max_depth, min_samples_split, min_samples_leaf               ║
║    max_features (RF: sqrt for clf, n/3 for reg)                 ║
║                                                                   ║
║  SINGLE TREE → RANDOM FOREST:                                   ║
║    + Bootstrap sampling (bagging)                                ║
║    + Random feature subsets at each split                       ║
║    + Majority vote → reduces variance                           ║
║                                                                   ║
║  PROS: No scaling needed, feature importance, handles mixed     ║
║  CONS: Single tree overfits, axis-aligned splits only           ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Entropy vs Gini? | Both measure impurity. Entropy uses log (slower), Gini uses squared probs. Similar results. sklearn defaults to Gini |
| 2 | How does Random Forest reduce overfitting? | Bagging (bootstrap) reduces variance. Feature sampling decorrelates trees |
| 3 | Calculate information gain? | IG = H(parent) - (n_left/n)*H(left) - (n_right/n)*H(right) |
| 4 | Decision tree vs Random Forest? | DT: interpretable but overfits. RF: more accurate, less interpretable, slower |
| 5 | Feature importance in RF? | Sum of impurity decrease weighted by samples reaching that node, averaged over trees |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Entropy H = -Σ p log₂ p, Gini G = 1 - Σ p²        │
## │  • IG = H(parent) - weighted H(children)               │
## │  • RF = bagging + feature sampling at each split        │
## │  • No scaling needed for trees                          │
## │  • feature_importances_ for feature selection           │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **KNN & Naive Bayes** — distance metrics, Bayes theorem